In [187]:
# Collin County Public Records Scraper

"""
This notebook scrapes probate records from Collin County's public search portal.

Target URL: https://collin.tx.publicsearch.us/results?department=RP&keywordSearch=false&limit=50&offset=0&recordedDateRange=18930107%2C20260123&searchOcrText=false&searchType=quickSearch&searchValue=probate&sort=desc&sortBy=recordedDate
"""

# Cell 1: Setup and Imports
import pandas as pd
import time
import json
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import requests
from bs4 import BeautifulSoup
import re

print("Libraries imported successfully!")

Libraries imported successfully!


In [188]:
# Cell 2: Configuration
# Base URL for Collin County public records
BASE_URL = "https://collin.tx.publicsearch.us"

# Search parameters
SEARCH_PARAMS = {
    "department": "RP",
    "keywordSearch": "false",
    "limit": "50",
    "offset": "0",
    "recordedDateRange": "18930107,20260123",
    "searchOcrText": "false",
    "searchType": "quickSearch",
    "searchValue": "probate",
    "sort": "desc",
    "sortBy": "recordedDate"
}

# Selenium configuration
CHROME_OPTIONS = Options()
CHROME_OPTIONS.add_argument("--headless")  # Run in background
CHROME_OPTIONS.add_argument("--no-sandbox")
CHROME_OPTIONS.add_argument("--disable-dev-shm-usage")
CHROME_OPTIONS.add_argument("--disable-gpu")
CHROME_OPTIONS.add_argument("--window-size=1920,1080")
CHROME_OPTIONS.add_argument("--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Wait times (in seconds)
PAGE_LOAD_WAIT = 10
ELEMENT_WAIT = 5

print("Configuration set up!")

Configuration set up!


In [189]:
# Cell 3: Initialize WebDriver
def initialize_driver():
    """Initialize Chrome WebDriver with configured options"""
    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=CHROME_OPTIONS)
        driver.set_page_load_timeout(30)
        print("WebDriver initialized successfully!")
        return driver
    except Exception as e:
        print(f"Error initializing WebDriver: {e}")
        return None

# Initialize the driver
driver = initialize_driver()

WebDriver initialized successfully!


In [190]:
# Cell 4: URL Builder and Page Loader
def build_search_url(params, offset=0):
    """Build search URL with given parameters and offset"""
    url_params = params.copy()
    url_params["offset"] = str(offset)
    
    query_string = "&".join([f"{k}={v}" for k, v in url_params.items()])
    return f"{BASE_URL}/results?{query_string}"

def load_page(driver, url, wait_time=PAGE_LOAD_WAIT):
    """Load a page and wait for it to complete"""
    try:
        print(f"Loading page: {url[:100]}...")
        driver.get(url)
        time.sleep(wait_time)
        
        # Wait for page to be ready
        WebDriverWait(driver, 25).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        
        print(f"Page loaded successfully. Title: {driver.title}")
        return True
        
    except Exception as e:
        print(f"Error loading page: {e}")
        return False

# Test URL building
test_url = build_search_url(SEARCH_PARAMS, offset=0)
print(f"Test URL: {test_url}")

Test URL: https://collin.tx.publicsearch.us/results?department=RP&keywordSearch=false&limit=50&offset=0&recordedDateRange=18930107,20260123&searchOcrText=false&searchType=quickSearch&searchValue=probate&sort=desc&sortBy=recordedDate


In [191]:
def has_more_pages(driver, current_offset, limit=50):
    """Check if there are more pages by looking for fewer results than expected"""
    try:
        # Count actual records on current page
        record_selectors = [
            ".record",
            ".result-item", 
            ".search-result",
            "tr.record-row",
            ".data-row",
            "[data-record]",
            ".result"
        ]
        
        actual_count = 0
        for selector in record_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    actual_count = len(elements)
                    break
            except:
                continue
        
        # If we got fewer results than the limit, we're likely at the end
        if actual_count < limit:
            print(f"Found {actual_count} records (less than limit {limit}), likely at end")
            return False
        
        # Additional check: look for "no results" or "end of results" messages
        no_results_selectors = [
            ".no-results",
            ".no-more-results",
            ".end-of-results",
            "[data-no-results]"
        ]
        
        for selector in no_results_selectors:
            try:
                element = driver.find_element(By.CSS_SELECTOR, selector)
                if element.is_displayed():
                    print("Found 'no more results' indicator")
                    return False
            except:
                continue
        
        return True
        
    except Exception as e:
        print(f"Error checking for more pages: {e}")
        return False

In [196]:
# Cell 6: Page Data Extraction Function
def get_total_results(driver):
    """Find and extract the total results count from the Search Result Totals span"""
    try:
        # Find the span element with aria-label="Search Result Totals"
        element = driver.find_element(By.CSS_SELECTOR, 'span[aria-label="Search Result Totals"]')
        
        if element:
            # Get the innerText
            inner_text = element.text.strip()
            print(f"Found Search Result Totals span: '{inner_text}'")
            
            # Look for the pattern "1-50 of 6,720 results for"
            # We need to extract the number after "of" and before "results"
            import re
            
            # Pattern to match "of X,XXX results"
            pattern = r'of\s+(\d{1,2},?\d{3})\s+results'
            match = re.search(pattern, inner_text, re.IGNORECASE)
            
            if match:
                total_text = match.group(1)  # This will be "6,720"
                total_results = int(total_text.replace(',', ''))  # Convert to 6720
                print(f"Total results found: {total_results:,}")
                return total_results
            else:
                # Alternative: split by "of" and "results"
                if "of" in inner_text and "results" in inner_text:
                    parts = inner_text.split("of")[1].split("results")[0].strip()
                    total_results = int(parts.replace(',', ''))
                    print(f"Total results found (alternative method): {total_results:,}")
                    return total_results
                else:
                    print(f"Could not find 'of X results' pattern in: {inner_text}")
                    return None
        else:
            print("Search Result Totals span element not found")
            return None
            
    except Exception as e:
        print(f"Error finding total results: {e}")
        return None

def extract_page_data(driver):
    """Extract all record data from the current page and return records array"""
    records = []
    
    try:
        # Get total results first
        total_results = get_total_results(driver)
        
        # First try to find table rows (most structured approach)
        tables = driver.find_elements(By.TAG_NAME, "table")
        for table in tables:
            rows = table.find_elements(By.TAG_NAME, "tr")
            if len(rows) > 2:  # Need at least header + 2 data rows
                print(f"Found table with {len(rows)} rows")
                
                # Skip first two rows (header row + empty row), process data rows
                for i, row in enumerate(rows[2:], 1):  # Skip first two rows
                    try:
                        # Extract specific fields using CSS selectors
                        record_data = {}
                        
                        # Grantor field - <td class="col-3" column="[object Object]"><span>
                        grantor_elem = row.find_element(By.CSS_SELECTOR, 'td.col-3[column="[object Object]"] span')
                        record_data['grantor'] = grantor_elem.text.strip() if grantor_elem else ""
                        
                        # Grantee field - <td class="col-4" column="[object Object]"><span>
                        grantee_elem = row.find_element(By.CSS_SELECTOR, 'td.col-4[column="[object Object]"] span')
                        record_data['grantee'] = grantee_elem.text.strip() if grantee_elem else ""
                        
                        # Doc Type field - <td class="col-5" column="[object Object]"><span><em>
                        doc_type_elem = row.find_element(By.CSS_SELECTOR, 'td.col-5[column="[object Object]"] span em')
                        record_data['doc_type'] = doc_type_elem.text.strip() if doc_type_elem else ""
                        
                        # Recorded Date field - <td class="col-6" column="[object Object]"><span>
                        recorded_date_elem = row.find_element(By.CSS_SELECTOR, 'td.col-6[column="[object Object]"] span')
                        record_data['recorded_date'] = recorded_date_elem.text.strip() if recorded_date_elem else ""
                        
                        # Doc Number field - <td class="col-7" column="[object Object]"><span>
                        doc_number_elem = row.find_element(By.CSS_SELECTOR, 'td.col-7[column="[object Object]"] span')
                        record_data['doc_number'] = doc_number_elem.text.strip() if doc_number_elem else ""
                        
                        # Book/Volume/Page field - <td class="col-8" column="[object Object]"><span>
                        book_volume_elem = row.find_element(By.CSS_SELECTOR, 'td.col-8[column="[object Object]"] span')
                        record_data['book_volume_page'] = book_volume_elem.text.strip() if book_volume_elem else ""
                        
                        # Legal Description field - <td class="col-9" column="[object Object]"><span class="css-1dro08k">
                        legal_desc_elem = row.find_element(By.CSS_SELECTOR, 'td.col-9')
                        record_data['legal_description'] = legal_desc_elem.text.strip() if legal_desc_elem else ""
                        
                        # Add metadata
                        record_data['record_number'] = i
                        record_data['extracted_at'] = datetime.now().isoformat()
                        
                        print(f"Row {i} - Grantor: {record_data['grantor'][:50]}...")
                        print(f"Row {i} - Recorded Date: {record_data['recorded_date']}")
                        
                        records.append(record_data)
                        
                    except Exception as e:
                        print(f"Error extracting data from row {i}: {e}")
                        # Continue with next row
                        continue
                
                if records:
                    break
        
        print(f"Extracted {len(records)} records from page")
        
        # Show sample of extracted data for debugging
        if records:
            print(f"Sample record keys: {list(records[0].keys())}")
            # Show first few records for verification
            for i, record in enumerate(records[:2]):
                print(f"Record {i+1}: {record}")
        
        return records
        
    except Exception as e:
        print(f"Error extracting page data: {e}")
        return []

In [197]:
# Cell 7: Main Scraping Function using Offset Pagination
def scrape_collin_county_records(max_pages=10, delay_between_pages=10):
    """Main scraping function using offset-based pagination"""
    all_records = []
    
    try:
        # Initialize driver
        driver = initialize_driver()
        if not driver:
            return []
        
        # Pagination settings
        limit = int(SEARCH_PARAMS["limit"])
        current_offset = 0
        page_num = 1
        
        while page_num <= max_pages:
            print(f"\n=== Scraping Page {page_num} (offset: {current_offset}) ===")
            
            # Build URL with current offset
            page_url = build_search_url(SEARCH_PARAMS, offset=current_offset)
            
            # Load page
            if not load_page(driver, page_url):
                print(f"Failed to load page {page_num}")
                break
            
            # Extract data from current page
            page_records = extract_page_data(driver)
            
            if page_records:
                # Add page number and offset to each record
                for record in page_records:
                    record['page_number'] = page_num
                    record['offset'] = current_offset
                
                all_records.extend(page_records)
                print(f"Extracted {len(page_records)} records from page {page_num}")
            else:
                print("No records found on this page")
                # If no records found, we might be at the end
                break
            
            # Check if there are more pages
            if not has_more_pages(driver, current_offset, limit):
                print("No more pages available")
                break
            
            # Increment offset for next page
            current_offset += limit
            page_num += 1
            
            # Wait between pages
            time.sleep(delay_between_pages)
        
        print(f"\n=== Scraping Complete ===")
        print(f"Total records scraped: {len(all_records)}")
        print(f"Total pages scraped: {page_num - 1}")
        print(f"Final offset: {current_offset}")
        
        return all_records
        
    except Exception as e:
        print(f"Error during scraping: {e}")
        return all_records
    
    finally:
        if 'driver' in locals():
            driver.quit()
            print("WebDriver closed")

In [198]:
# Cell 8: Data Processing and Export
def process_and_export_data(records, filename_prefix="collin_county_probate"):
    """Process scraped data and export to CSV format"""
    if not records:
        print("No records to process")
        return
    
    # Convert to DataFrame
    df = pd.DataFrame(records)
    
    # Add processing timestamp
    df['processed_at'] = datetime.now().isoformat()
    
    # Display basic statistics
    print(f"\n=== Data Summary ===")
    print(f"Total records: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nFirst few records:")
    display(df.head())
    
    # Create data directory if it doesn't exist
    import os
    data_dir = "data"
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
        print(f"Created directory: {data_dir}")
    
    # Generate date-based filename in data folder
    csv_filename = os.path.join(data_dir, f"{datetime.now().strftime('%Y-%m-%d')}-probate-records.csv")
    
    # Export to CSV
    df.to_csv(csv_filename, index=False)
    print(f"\nData exported to: {csv_filename}")
    
    return df

In [200]:
# Cell 9: Run the Scraper
# Configure scraping parameters
MAX_PAGES = 5  # Adjust as needed
DELAY_BETWEEN_PAGES = 3  # Seconds between requests

print("Starting Collin County probate records scraper...")
print(f"Max pages to scrape: {MAX_PAGES}")
print(f"Delay between pages: {DELAY_BETWEEN_PAGES} seconds")
print(f"Using offset-based pagination with limit: {SEARCH_PARAMS['limit']}")

# Run the scraper
scraped_records = scrape_collin_county_records(
    max_pages=MAX_PAGES,
    delay_between_pages=DELAY_BETWEEN_PAGES
)

df = process_and_export_data(scraped_records)
df

Starting Collin County probate records scraper...
Max pages to scrape: 5
Delay between pages: 3 seconds
Using offset-based pagination with limit: 50
WebDriver initialized successfully!

=== Scraping Page 1 (offset: 0) ===
Loading page: https://collin.tx.publicsearch.us/results?department=RP&keywordSearch=false&limit=50&offset=0&record...
Page loaded successfully. Title: Search Results
Found Search Result Totals span: '1-50 of 6,720 results for'
Total results found: 6,720
Found table with 52 rows
Row 1 - Grantor: GOYAL MADAN DECEASED ESTATE...
Row 1 - Recorded Date: 1/23/2026
Row 2 - Grantor: CHERRY ERIKA WOOD DECEASED ESTATE...
Row 2 - Recorded Date: 1/23/2026
Row 3 - Grantor: KING L M JR DECEASED ESTATE...
Row 3 - Recorded Date: 1/21/2026
Row 4 - Grantor: KIM JUNGJI DECEASED ESTATE...
Row 4 - Recorded Date: 1/21/2026
Row 5 - Grantor: HINCKLEY BONNIE YVONNE DECEASED ESTATE...
Row 5 - Recorded Date: 1/20/2026
Row 6 - Grantor: ROYAL COKE DECEASED ESTATE...
Row 6 - Recorded Date: 1/16/202

,grantor,grantee,doc_type,recorded_date,doc_number,book_volume_page,legal_description,record_number,extracted_at,page_number,offset,processed_at
0,GOYAL MADAN DECEASED ESTATE,PUBLIC,PROBATE,1/23/2026,2026000009450,--/--/--,N/A,1,2026-01-29T20:00:15.944810,1,0,2026-01-29T20:00:18.800686
1,CHERRY ERIKA WOOD DECEASED ESTATE,PUBLIC,PROBATE,1/23/2026,2026000009280,--/--/--,N/A,2,2026-01-29T20:00:15.989922,1,0,2026-01-29T20:00:18.800686
2,KING L M JR DECEASED ESTATE,PUBLIC,PROBATE,1/21/2026,2026000008491,--/--/--,N/A,3,2026-01-29T20:00:16.036722,1,0,2026-01-29T20:00:18.800686
3,KIM JUNGJI DECEASED ESTATE,PUBLIC,PROBATE,1/21/2026,2026000008179,--/--/--,Subdivision - Name: SMITH ESTATES FRISCO Lot: ...,4,2026-01-29T20:00:16.089897,1,0,2026-01-29T20:00:18.800686
4,HINCKLEY BONNIE YVONNE DECEASED ESTATE,PUBLIC,PROBATE,1/20/2026,2026000007253,--/--/--,N/A,5,2026-01-29T20:00:16.476293,1,0,2026-01-29T20:00:18.800686



Data exported to: data/2026-01-29-probate-records.csv


,grantor,grantee,doc_type,recorded_date,doc_number,book_volume_page,legal_description,record_number,extracted_at,page_number,offset,processed_at
0,GOYAL MADAN DECEASED ESTATE,PUBLIC,PROBATE,1/23/2026,2026000009450,--/--/--,N/A,1,2026-01-29T20:00:15.944810,1,0,2026-01-29T20:00:18.800686
1,CHERRY ERIKA WOOD DECEASED ESTATE,PUBLIC,PROBATE,1/23/2026,2026000009280,--/--/--,N/A,2,2026-01-29T20:00:15.989922,1,0,2026-01-29T20:00:18.800686
2,KING L M JR DECEASED ESTATE,PUBLIC,PROBATE,1/21/2026,2026000008491,--/--/--,N/A,3,2026-01-29T20:00:16.036722,1,0,2026-01-29T20:00:18.800686
3,KIM JUNGJI DECEASED ESTATE,PUBLIC,PROBATE,1/21/2026,2026000008179,--/--/--,Subdivision - Name: SMITH ESTATES FRISCO Lot: ...,4,2026-01-29T20:00:16.089897,1,0,2026-01-29T20:00:18.800686
4,HINCKLEY BONNIE YVONNE DECEASED ESTATE,PUBLIC,PROBATE,1/20/2026,2026000007253,--/--/--,N/A,5,2026-01-29T20:00:16.476293,1,0,2026-01-29T20:00:18.800686
5,ROYAL COKE DECEASED ESTATE,PUBLIC,PROBATE,1/16/2026,2026000006940,--/--/--,Subdivision - Name: MARTIN FARMS ST PAUL Lot: ...,6,2026-01-29T20:00:16.579838,1,0,2026-01-29T20:00:18.800686
6,MALONE MARK ALLAN DECEASED ESTATE,PUBLIC,PROBATE,1/16/2026,2026000006343,--/--/--,N/A,7,2026-01-29T20:00:16.638556,1,0,2026-01-29T20:00:18.800686
7,CADY ARMANDINA RUIZ DECEASED ESTATE,PUBLIC,PROBATE,1/13/2026,2026000004474,--/--/--,N/A,8,2026-01-29T20:00:16.688325,1,0,2026-01-29T20:00:18.800686
8,BLOYS BARBARA JANE WEATHERS DECEASED ESTATE,PUBLIC,PROBATE,1/13/2026,2026000004383,--/--/--,N/A,9,2026-01-29T20:00:16.736943,1,0,2026-01-29T20:00:18.800686
9,WAGNER CYNTHIA DIANE DECEASED ESTATE,PUBLIC,PROBATE,1/13/2026,2026000004382,--/--/--,N/A,10,2026-01-29T20:00:16.786405,1,0,2026-01-29T20:00:18.800686


In [186]:
df.columns

Index(['plain_text', 'html_content', 'record_number', 'extracted_at'], dtype='str')